# Gene subselection strategy

The idea behind this gene‑subselection strategy is to boil the full ∼19 k‑gene expression matrix down to a compact, biologically meaningful panel that best separates the two PAM50 subtypes of interest (Basal‑like vs. Luminal A). First, low‑expressed genes that are detected in fewer than a user‑defined number of samples are removed, which reduces noise and prevents spurious signals. Then, for each remaining gene a very fast ordinary‑least‑squares regression is fitted with the binary phenotype (1 = Basal, 0 = LumA) as the sole predictor; the regression coefficient (β₁) serves as an estimate of the log‑fold‑change between the two groups, and its associated t‑test p‑value quantifies how unlikely such a difference would arise by chance. All p‑values are corrected for multiple testing using the Benjamini–Hochberg false‑discovery‑rate (FDR) procedure. To combine effect size and significance into a single ranking metric, each gene receives a “score” equal to |logFC| × –log₁₀(FDR), so genes that change a lot and are highly significant attain the highest scores. The genes are then sorted by this score and the top N (e.g., 1 024 or 2 048) are retained; a list of classic PAM50 genes can be force‑included to guarantee the canonical markers remain present. Optionally, highly correlated genes (pairwise Pearson ρ > 0.9) are pruned so that redundant features do not inflate the dimensionality. The final output is a reduced set of ≈1 k–2 k non‑redundant genes that capture the transcriptional contrast between Basal‑like and Luminal A tumors and can be used for downstream analyses such as UMAP visualization, classification, or as conditioning input for a diffusion‑based image‑generation model.

In [20]:
import pandas as pd
import numpy as np
import scanpy as sc               
import umap
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import plotly.express as px

In [ ]:
input_csv_path = "../../data/brca_gene_expression_with_subtypes.csv"
target_gene_number = 1024                 # or 2048 – number of genes to finally keep
min_nonzero = 5                    # keep a gene if it is >0 in >= min_nonzero samples
corr_threshold = 0.90                 # set to None to skip redundancy filtering
force_pam50 = True                 # keep the 50 PAM50 genes even if they are not top‑ranked
pam50_path = "../../data/pam50_gene_list.txt"   

# Load data
df = pd.read_csv(input_csv_path)

patient_col = "Patient_ID"
subtype_col = "Majority_Subtype_mRNA"

df.set_index(patient_col, inplace=True)          # use patient ID as index

# Keep only Basal and LumA (ignore LumB, Her2, Normal)
wanted = ["Basal", "LumA"]
df = df.loc[df[subtype_col].isin(wanted)].copy()

# Build the binary target vector
# Basal = 1, LumA = 0 
y = (df[subtype_col] == "Basal").astype(int).values

# Keep the pure expression matrix (drop the subtype column)
expr = df.drop(columns=[subtype_col])

#  Load PAM50 list to force‑include later
if force_pam50:
    pam50_genes = pd.read_csv(pam50_path, header=None).iloc[:, 0].astype(str).tolist()
else:
    pam50_genes = []

# Basic filtering – drop low‑expressed genes
mask = (expr > 0).sum(axis=0) >= min_nonzero
expr = expr.loc[:, mask]

# Fast “pseudo‑DE” (OLS per gene)
def ols_de(gene_series):
    """Return (logFC, p‑value) from OLS: expr ~ intercept + label."""
    model = sm.OLS(gene_series.values, sm.add_constant(y)).fit()
    return model.params[1], model.pvalues[1]

logfc, pval = zip(*[ols_de(expr[g]) for g in expr.columns])

de_df = pd.DataFrame({
    "gene": expr.columns,
    "logFC": np.asarray(logfc),
    "pval":  np.asarray(pval)
})
de_df["adj_p"] = multipletests(de_df["pval"], method="fdr_bh")[1]
de_df["score"] = np.abs(de_df["logFC"]) * -np.log10(de_df["adj_p"] + 1e-300)

# Select top N genes + force‑include PAM50
top_genes = de_df.sort_values("score", ascending=False)["gene"].tolist()[:target_gene_number]
selected = list(set(top_genes + pam50_genes))

# Optional redundancy removal (highly correlated genes)
if corr_threshold is not None:
    X = expr[selected].values
    corr_mat = np.corrcoef(X, rowvar=False)

    keep_idx = []
    for i, _ in enumerate(selected):
        if any(np.abs(corr_mat[i, j]) > corr_threshold for j in keep_idx):
            continue
        keep_idx.append(i)

    selected = [selected[i] for i in keep_idx]

# UMAP sanity check (optional)
scaled_X = sc.pp.scale(expr[selected].values)          # z‑score each gene
umap_emb = umap.UMAP(random_state=42,
                    n_neighbors=30,
                    min_dist=0.2).fit_transform(scaled_X)

plt.figure(figsize=(6,5))
plt.scatter(umap_emb[:,0], umap_emb[:,1], c=y,
            cmap="coolwarm", s=30, edgecolor="k", linewidth=0.2)
plt.title(f"UMAP of {len(selected)} selected genes (Basal vs LumA)")
plt.xlabel("UMAP‑1")
plt.ylabel("UMAP‑2")
plt.tight_layout()
plt.show()

# Save the final gene list (optional)
pd.Series(selected, name="gene").to_csv(
    f"selected_genes_{len(selected)}_genes.txt", index=False, header=False)

In [5]:
# read in txt file with list of genes
with open('../../data/selected_genes_1026_genes.txt', 'r') as file:
    extended_pam50_genes = [line.strip() for line in file.readlines()]

In [10]:
# filter the columns of brca_df to only include pam50 genes
df = df.reset_index()  
brca_extended_pam50_df = df[['Patient_ID', 'Majority_Subtype_mRNA'] + extended_pam50_genes]

In [19]:
# Preprocess data
selected_data = brca_extended_pam50_df.drop(columns=['Patient_ID', 'Majority_Subtype_mRNA'])
scaled_X = sc.pp.scale(selected_data.values)  # z-score each gene

# UMAP with same parameters
reducer = umap.UMAP(random_state=42, n_neighbors=30, min_dist=0.2)
embedding = reducer.fit_transform(scaled_X)

# Create a dataframe with the UMAP coordinates
umap_df = pd.DataFrame({'UMAP1': embedding[:, 0], 'UMAP2': embedding[:, 1], 
                        'Majority_Subtype_mRNA': brca_extended_pam50_df['Majority_Subtype_mRNA']})

# Plot using plotly
fig = px.scatter(umap_df, x='UMAP1', y='UMAP2', color='Majority_Subtype_mRNA',
                 title='UMAP of BRCA Gene Expression Colored by PAM50 Subtypes')
fig.show()

/home/maralampert/micromamba/envs/rnaseq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [22]:
# save plot to html
umap_df.to_html('../plots/umap_brca_basal_vs_luma.html', index=False)

Z-scoring is a method of standardizing data by subtracting the mean and dividing by the standard deviation for each feature. This process, also known as standardization or z-normalization, centers the data around zero and scales it to have a standard deviation of 1. By doing so, z-scoring prevents features with large ranges from dominating the analysis and improves the interpretability of the results. The resulting z-scores represent the number of standard deviations away from the mean that each original value is, allowing for more robust and comparable analysis across different features.